In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import os
import numpy as np


class PatchGANDiscriminator(nn.Module):
    """
    PatchGAN Discriminator for phase images.
    Classifies 70x70 patches as real or fake.
    Better for texture and local structure discrimination.
    """
    def __init__(self, in_channels=3, ndf=64):
        """
        Args:
            in_channels: Number of input channels (3 for RGB)
            ndf: Number of discriminator filters in first conv layer
        """
        super(PatchGANDiscriminator, self).__init__()
        
        # 512x512 -> 256x256
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels, ndf, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 256x256 -> 128x128
        self.layer2 = nn.Sequential(
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 128x128 -> 64x64
        self.layer3 = nn.Sequential(
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 64x64 -> 32x32
        self.layer4 = nn.Sequential(
            nn.Conv2d(ndf * 4, ndf * 8, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 32x32 -> 31x31 (PatchGAN output)
        self.layer5 = nn.Sequential(
            nn.Conv2d(ndf * 8, 1, kernel_size=4, stride=1, padding=1),
            # No sigmoid here - will use BCEWithLogitsLoss
        )
        
    def forward(self, x):
        """
        Args:
            x: Input image tensor [B, 3, 512, 512]
        Returns:
            Patch predictions [B, 1, 31, 31]
        """
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        return x


class ConditionalDiscriminator(nn.Module):
    """
    Conditional Discriminator that takes both real/fake image and a condition.
    Useful for comparing U-Net outputs with originals.
    """
    def __init__(self, in_channels=3, condition_channels=3, ndf=64):
        """
        Args:
            in_channels: Channels in image to classify
            condition_channels: Channels in conditioning input
            ndf: Base number of filters
        """
        super(ConditionalDiscriminator, self).__init__()
        
        # Concatenate image and condition
        total_channels = in_channels + condition_channels
        
        # 512x512 -> 256x256
        self.layer1 = nn.Sequential(
            nn.Conv2d(total_channels, ndf, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 256x256 -> 128x128
        self.layer2 = nn.Sequential(
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 128x128 -> 64x64
        self.layer3 = nn.Sequential(
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 64x64 -> 32x32
        self.layer4 = nn.Sequential(
            nn.Conv2d(ndf * 4, ndf * 8, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 32x32 -> 31x31
        self.layer5 = nn.Sequential(
            nn.Conv2d(ndf * 8, 1, kernel_size=4, stride=1, padding=1),
        )
        
    def forward(self, x, condition):
        """
        Args:
            x: Image to classify [B, 3, 512, 512]
            condition: Conditioning input [B, 3, 512, 512]
        Returns:
            Patch predictions [B, 1, 31, 31]
        """
        # Concatenate along channel dimension
        x = torch.cat([x, condition], dim=1)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        return x


class DeepDiscriminator(nn.Module):
    """
    Deeper discriminator with more layers for better feature learning.
    Single scalar output per image.
    """
    def __init__(self, in_channels=3, ndf=64):
        super(DeepDiscriminator, self).__init__()
        
        self.features = nn.Sequential(
            # 512x512 -> 256x256
            nn.Conv2d(in_channels, ndf, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 256x256 -> 128x128
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 128x128 -> 64x64
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 64x64 -> 32x32
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 32x32 -> 16x16
            nn.Conv2d(ndf * 8, ndf * 8, 4, 2, 1),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 16x16 -> 8x8
            nn.Conv2d(ndf * 8, ndf * 8, 4, 2, 1),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
        )
        
        # Global average pooling + FC
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(ndf * 8, 1)
        )
        
    def forward(self, x):
        """
        Args:
            x: Input image [B, 3, 512, 512]
        Returns:
            Scalar prediction [B, 1]
        """
        x = self.features(x)
        x = self.classifier(x)
        return x


class SpectralNormDiscriminator(nn.Module):
    """
    Discriminator with Spectral Normalization for training stability.
    Recommended for high-quality image generation tasks.
    """
    def __init__(self, in_channels=3, ndf=64):
        super(SpectralNormDiscriminator, self).__init__()
        
        # 512x512 -> 256x256
        self.layer1 = nn.Sequential(
            nn.utils.spectral_norm(
                nn.Conv2d(in_channels, ndf, kernel_size=4, stride=2, padding=1)
            ),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 256x256 -> 128x128
        self.layer2 = nn.Sequential(
            nn.utils.spectral_norm(
                nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1)
            ),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 128x128 -> 64x64
        self.layer3 = nn.Sequential(
            nn.utils.spectral_norm(
                nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1)
            ),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 64x64 -> 32x32
        self.layer4 = nn.Sequential(
            nn.utils.spectral_norm(
                nn.Conv2d(ndf * 4, ndf * 8, kernel_size=4, stride=2, padding=1)
            ),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # 32x32 -> 16x16
        self.layer5 = nn.Sequential(
            nn.utils.spectral_norm(
                nn.Conv2d(ndf * 8, ndf * 8, kernel_size=4, stride=2, padding=1)
            ),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # Global pooling and classification
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.utils.spectral_norm(nn.Linear(ndf * 8, 1))
        )
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.classifier(x)
        return x


# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

class DiscriminatorLoss(nn.Module):
    """
    Standard GAN discriminator loss with BCEWithLogitsLoss.
    """
    def __init__(self):
        super(DiscriminatorLoss, self).__init__()
        self.criterion = nn.BCEWithLogitsLoss()
        
    def forward(self, real_output, fake_output):
        """
        Args:
            real_output: Discriminator output for real images
            fake_output: Discriminator output for fake/generated images
        Returns:
            Discriminator loss
        """
        # Real images should be classified as 1
        real_labels = torch.ones_like(real_output)
        real_loss = self.criterion(real_output, real_labels)
        
        # Fake images should be classified as 0
        fake_labels = torch.zeros_like(fake_output)
        fake_loss = self.criterion(fake_output, fake_labels)
        
        # Total discriminator loss
        d_loss = (real_loss + fake_loss) / 2
        
        return d_loss


class WassersteinLoss(nn.Module):
    """
    Wasserstein GAN discriminator loss (also called critic loss).
    More stable training, no sigmoid in discriminator.
    """
    def __init__(self):
        super(WassersteinLoss, self).__init__()
        
    def forward(self, real_output, fake_output):
        """
        Args:
            real_output: Discriminator output for real images
            fake_output: Discriminator output for fake images
        Returns:
            Wasserstein loss
        """
        # Maximize: E[D(real)] - E[D(fake)]
        # Minimize: -E[D(real)] + E[D(fake)]
        d_loss = -torch.mean(real_output) + torch.mean(fake_output)
        return d_loss


class HingeLoss(nn.Module):
    """
    Hinge loss for discriminator (used in modern GANs like BigGAN).
    Often provides more stable training.
    """
    def __init__(self):
        super(HingeLoss, self).__init__()
        
    def forward(self, real_output, fake_output):
        """
        Args:
            real_output: Discriminator output for real images
            fake_output: Discriminator output for fake images
        Returns:
            Hinge loss
        """
        # D(real) should be > 1, D(fake) should be < -1
        real_loss = torch.mean(F.relu(1.0 - real_output))
        fake_loss = torch.mean(F.relu(1.0 + fake_output))
        
        d_loss = real_loss + fake_loss
        return d_loss


class LeastSquaresLoss(nn.Module):
    """
    Least Squares GAN (LSGAN) loss.
    Often more stable than standard GAN loss.
    """
    def __init__(self):
        super(LeastSquaresLoss, self).__init__()
        
    def forward(self, real_output, fake_output):
        """
        Args:
            real_output: Discriminator output for real images
            fake_output: Discriminator output for fake images
        Returns:
            Least squares loss
        """
        # Real images should output 1
        real_loss = torch.mean((real_output - 1) ** 2)
        
        # Fake images should output 0
        fake_loss = torch.mean(fake_output ** 2)
        
        d_loss = (real_loss + fake_loss) / 2
        return d_loss


class RelativisticAverageLoss(nn.Module):
    """
    Relativistic Average GAN loss.
    Compares real and fake predictions relativistically.
    """
    def __init__(self):
        super(RelativisticAverageLoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()
        
    def forward(self, real_output, fake_output):
        """
        Args:
            real_output: Discriminator output for real images
            fake_output: Discriminator output for fake images
        Returns:
            Relativistic loss
        """
        # Real should be more real than average of fakes
        real_loss = self.bce(
            real_output - torch.mean(fake_output),
            torch.ones_like(real_output)
        )
        
        # Fake should be less real than average of reals
        fake_loss = self.bce(
            fake_output - torch.mean(real_output),
            torch.zeros_like(fake_output)
        )
        
        d_loss = (real_loss + fake_loss) / 2
        return d_loss


# ============================================================================
# GRADIENT PENALTY (for WGAN-GP)
# ============================================================================

def compute_gradient_penalty(discriminator, real_images, fake_images, device):
    """
    Compute gradient penalty for WGAN-GP.
    
    Args:
        discriminator: Discriminator network
        real_images: Real image batch
        fake_images: Fake image batch
        device: torch device
    Returns:
        Gradient penalty value
    """
    batch_size = real_images.size(0)
    
    # Random weight term for interpolation
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    
    # Interpolate between real and fake images
    interpolates = (alpha * real_images + (1 - alpha) * fake_images).requires_grad_(True)
    
    # Get discriminator output for interpolated images
    d_interpolates = discriminator(interpolates)
    
    # Get gradients w.r.t. interpolated images
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(d_interpolates),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    
    # Flatten gradients
    gradients = gradients.view(batch_size, -1)
    
    # Compute gradient penalty
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    
    return gradient_penalty


# ============================================================================
# DATASET
# ============================================================================

class PhaseImageDataset(Dataset):
    """
    Dataset for real and generated/reconstructed phase images.
    """
    def __init__(self, real_dir, fake_dir, transform=None):
        """
        Args:
            real_dir: Directory containing real/original phase images
            fake_dir: Directory containing generated/U-Net reconstructed images
            transform: Image transformations
        """
        self.real_dir = real_dir
        self.fake_dir = fake_dir
        
        # Get matching files
        real_files = set([f for f in os.listdir(real_dir) 
                         if f.endswith(('.png', '.jpg', '.tif', '.tiff'))])
        fake_files = set([f for f in os.listdir(fake_dir) 
                         if f.endswith(('.png', '.jpg', '.tif', '.tiff'))])
        
        # Only use files that exist in both directories
        self.image_files = list(real_files.intersection(fake_files))
        
        if len(self.image_files) == 0:
            raise ValueError("No matching files found in real and fake directories!")
        
        print(f"Found {len(self.image_files)} matching image pairs")
        
        # Default transform
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((512, 512)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                   std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transform
            
    def __len__(self):
        return len(self.image_files)
    
    def load_image(self, img_path):
        """Load and convert image to RGB."""
        img = Image.open(img_path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return img
    
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        
        real_path = os.path.join(self.real_dir, img_name)
        fake_path = os.path.join(self.fake_dir, img_name)
        
        real_img = self.load_image(real_path)
        fake_img = self.load_image(fake_path)
        
        if self.transform:
            real_img = self.transform(real_img)
            fake_img = self.transform(fake_img)
        
        return real_img, fake_img, img_name


# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================

def train_discriminator_epoch(discriminator, dataloader, criterion, optimizer, 
                              device, use_gp=False, gp_lambda=10):
    """
    Train discriminator for one epoch.
    
    Args:
        discriminator: Discriminator model
        dataloader: DataLoader with real and fake image pairs
        criterion: Loss function
        optimizer: Optimizer
        device: torch device
        use_gp: Whether to use gradient penalty (for WGAN-GP)
        gp_lambda: Gradient penalty coefficient
    """
    discriminator.train()
    running_loss = 0.0
    running_gp = 0.0
    
    # Metrics
    real_accuracy = 0.0
    fake_accuracy = 0.0
    total_samples = 0
    
    for batch_idx, (real_imgs, fake_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        fake_imgs = fake_imgs.to(device)
        batch_size = real_imgs.size(0)
        
        optimizer.zero_grad()
        
        # Forward pass
        real_output = discriminator(real_imgs)
        fake_output = discriminator(fake_imgs.detach())  # Don't backprop through generator
        
        # Compute loss
        d_loss = criterion(real_output, fake_output)
        
        # Add gradient penalty if using WGAN-GP
        if use_gp:
            gp = compute_gradient_penalty(discriminator, real_imgs, fake_imgs, device)
            d_loss = d_loss + gp_lambda * gp
            running_gp += gp.item()
        
        # Backward pass
        d_loss.backward()
        optimizer.step()
        
        running_loss += d_loss.item()
        
        # Calculate accuracy (for binary classification)
        if isinstance(criterion, (DiscriminatorLoss, LeastSquaresLoss, RelativisticAverageLoss)):
            with torch.no_grad():
                real_pred = torch.sigmoid(real_output.mean(dim=[1, 2, 3]) if len(real_output.shape) == 4 else real_output)
                fake_pred = torch.sigmoid(fake_output.mean(dim=[1, 2, 3]) if len(fake_output.shape) == 4 else fake_output)
                
                real_correct = (real_pred > 0.5).sum().item()
                fake_correct = (fake_pred < 0.5).sum().item()
                
                real_accuracy += real_correct
                fake_accuracy += fake_correct
                total_samples += batch_size
        
        if (batch_idx + 1) % 10 == 0:
            avg_loss = running_loss / (batch_idx + 1)
            if use_gp:
                avg_gp = running_gp / (batch_idx + 1)
                print(f'Batch [{batch_idx+1}/{len(dataloader)}], '
                      f'Loss: {avg_loss:.4f}, GP: {avg_gp:.4f}')
            else:
                print(f'Batch [{batch_idx+1}/{len(dataloader)}], '
                      f'Loss: {avg_loss:.4f}')
    
    epoch_loss = running_loss / len(dataloader)
    
    if total_samples > 0:
        real_acc = 100.0 * real_accuracy / total_samples
        fake_acc = 100.0 * fake_accuracy / total_samples
        overall_acc = (real_accuracy + fake_accuracy) / (2 * total_samples) * 100.0
        return epoch_loss, real_acc, fake_acc, overall_acc
    else:
        return epoch_loss, 0.0, 0.0, 0.0


def validate_discriminator(discriminator, dataloader, criterion, device):
    """
    Validate discriminator.
    """
    discriminator.eval()
    running_loss = 0.0
    real_accuracy = 0.0
    fake_accuracy = 0.0
    total_samples = 0
    
    with torch.no_grad():
        for real_imgs, fake_imgs, _ in dataloader:
            real_imgs = real_imgs.to(device)
            fake_imgs = fake_imgs.to(device)
            batch_size = real_imgs.size(0)
            
            # Forward pass
            real_output = discriminator(real_imgs)
            fake_output = discriminator(fake_imgs)
            
            # Compute loss
            d_loss = criterion(real_output, fake_output)
            running_loss += d_loss.item()
            
            # Calculate accuracy
            if isinstance(criterion, (DiscriminatorLoss, LeastSquaresLoss, RelativisticAverageLoss)):
                real_pred = torch.sigmoid(real_output.mean(dim=[1, 2, 3]) if len(real_output.shape) == 4 else real_output)
                fake_pred = torch.sigmoid(fake_output.mean(dim=[1, 2, 3]) if len(fake_output.shape) == 4 else fake_output)
                
                real_correct = (real_pred > 0.5).sum().item()
                fake_correct = (fake_pred < 0.5).sum().item()
                
                real_accuracy += real_correct
                fake_accuracy += fake_correct
                total_samples += batch_size
    
    val_loss = running_loss / len(dataloader)
    
    if total_samples > 0:
        real_acc = 100.0 * real_accuracy / total_samples
        fake_acc = 100.0 * fake_accuracy / total_samples
        overall_acc = (real_accuracy + fake_accuracy) / (2 * total_samples) * 100.0
        return val_loss, real_acc, fake_acc, overall_acc
    else:
        return val_loss, 0.0, 0.0, 0.0


def main():
    """
    Main training function for discriminator.
    """
    # Hyperparameters
    BATCH_SIZE = 16
    LEARNING_RATE = 0.0002
    BETA1 = 0.5  # Adam beta1 parameter
    BETA2 = 0.999
    NUM_EPOCHS = 100
    
    # Paths
    REAL_DIR = '/path/to/real/phase/images'
    FAKE_DIR = '/path/to/generated/or/unet/images'
    
    # Choose discriminator type
    # Options: 'patchgan', 'deep', 'spectral', 'conditional'
    DISCRIMINATOR_TYPE = 'patchgan'
    
    # Choose loss type
    # Options: 'standard', 'wasserstein', 'hinge', 'lsgan', 'relativistic'
    LOSS_TYPE = 'standard'
    
    # Gradient penalty (only for Wasserstein)
    USE_GP = False
    GP_LAMBDA = 10
    
    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    
    # Create dataset
    dataset = PhaseImageDataset(REAL_DIR, FAKE_DIR)
    
    # Split into train/val
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    
    # Initialize discriminator
    if DISCRIMINATOR_TYPE == 'patchgan':
        discriminator = PatchGANDiscriminator(in_channels=3, ndf=64).to(device)
    elif DISCRIMINATOR_TYPE == 'deep':
        discriminator = DeepDiscriminator(in_channels=3, ndf=64).to(device)
    elif DISCRIMINATOR_TYPE == 'spectral':
        discriminator = SpectralNormDiscriminator(in_channels=3, ndf=64).to(device)
    elif DISCRIMINATOR_TYPE == 'conditional':
        discriminator = ConditionalDiscriminator(in_channels=3, condition_channels=3, ndf=64).to(device)
    else:
        raise ValueError(f"Unknown discriminator type: {DISCRIMINATOR_TYPE}")
    
    print(f"Using {DISCRIMINATOR_TYPE} discriminator")
    
    # Initialize loss
    if LOSS_TYPE == 'standard':
        criterion = DiscriminatorLoss()
    elif LOSS_TYPE == 'wasserstein':
        criterion = WassersteinLoss()
        USE_GP = True
    elif LOSS_TYPE == 'hinge':
        criterion = HingeLoss()
    elif LOSS_TYPE == 'lsgan':
        criterion = LeastSquaresLoss()
    elif LOSS_TYPE == 'relativistic':
        criterion = RelativisticAverageLoss()
    else:
        raise ValueError(f"Unknown loss type: {LOSS_TYPE}")
    
    print(f"Using {LOSS_TYPE} loss")
    
    # Optimizer
    optimizer = torch.optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, 
                                betas=(BETA1, BETA2))
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, verbose=True
    )
    
    # Training loop
    best_val_loss = float('inf')
    
    for epoch in range(NUM_EPOCHS):
        print(f'\nEpoch [{epoch+1}/{NUM_EPOCHS}]')
        print('-' * 70)
        
        # Train
        train_loss, train_real_acc, train_fake_acc, train_overall_acc = train_discriminator_epoch(
            discriminator, train_loader, criterion, optimizer, device, USE_GP, GP_LAMBDA
        )
        
        print(f'Train Loss: {train_loss:.4f}')
        print(f'Train Real Accuracy: {train_real_acc:.2f}%')
        print(f'Train Fake Accuracy: {train_fake_acc:.2f}%')
        print(f'Train Overall Accuracy: {train_overall_acc:.2f}%')
        
        # Validate
        val_loss, val_real_acc, val_fake_acc, val_overall_acc = validate_discriminator(
            discriminator, val_loader, criterion, device
        )
        
        print(f'Val Loss: {val_loss:.4f}')
        print(f'Val Real Accuracy: {val_real_acc:.2f}%')
        print(f'Val Fake Accuracy: {val_fake_acc:.2f}%')
        print(f'Val Overall Accuracy: {val_overall_acc:.2f}%')
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'discriminator_state_dict': discriminator.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_accuracy': val_overall_acc,
                'discriminator_type': DISCRIMINATOR_TYPE,
                'loss_type': LOSS_TYPE,
            }, 'best_discriminator.pth')
            print(f'Saved best model with val_loss: {val_loss:.4f}')
    
    print('\nTraining completed!')


if __name__ == '__main__':
    main()